# Clase 224 — Métricas de fairness: DP, equalized odds, calibration

Dataset sintético binario con atributo protegido `A∈{0,1}` y **base rates diferentes** (necesario para activar el teorema de imposibilidad). Requiere: `pip install numpy pandas scikit-learn matplotlib`. `fairlearn` opcional — implementamos todo a mano.

## 🧠 Intuición previa

**No hay UNA definición de "justo".** *Demographic parity* (misma tasa de aprobación por
grupo), *equalized odds* (mismos TPR y FPR por grupo) y *calibration* (un score de 0.7
significa lo mismo en todos los grupos) son tres nociones **razonables** y, cuando las tasas
base difieren entre grupos, **matemáticamente incompatibles**: no podés satisfacer las tres
a la vez (Kleinberg-Mullainathan-Raghavan 2016, Chouldechova 2017). La pregunta no es "¿es
justo?" sino **"¿cuál de estas justicias importa en mi caso, y qué estoy dispuesto a sacrificar?"**

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 10_000

# Atributo protegido (50/50)
A = rng.integers(0, 2, n)

# Features con distribución que depende de A (proxy realista)
x1 = rng.normal(loc=0.5 * A, scale=1.0, size=n)
x2 = rng.normal(loc=-0.3 * A, scale=1.0, size=n)
x3 = rng.normal(loc=0.0, scale=1.0, size=n)

# Base rates DIFERENTES → activa impossibility
logits = 1.2 * x1 - 0.8 * x2 + 0.5 * x3 + np.where(A == 0, 0.4, -0.4)
p = 1 / (1 + np.exp(-logits))
y = (rng.random(n) < p).astype(int)

X = np.column_stack([x1, x2, x3, A])
print(f'n={n} | base rate A=0: {y[A==0].mean():.3f} | base rate A=1: {y[A==1].mean():.3f}')

## 1. Baseline: LogisticRegression

In [ ]:
X_tr, X_te, y_tr, y_te, A_tr, A_te = train_test_split(
    X, y, A, test_size=0.3, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
s = clf.predict_proba(X_te)[:, 1]
yhat = (s >= 0.5).astype(int)

print(f'Accuracy global: {accuracy_score(y_te, yhat):.4f}')
print(f'AUC:             {roc_auc_score(y_te, s):.4f}')
print(f'Acc A=0: {accuracy_score(y_te[A_te==0], yhat[A_te==0]):.4f}')
print(f'Acc A=1: {accuracy_score(y_te[A_te==1], yhat[A_te==1]):.4f}')

## 2. Demographic parity gap

`DP_gap = |P(Ŷ=1|A=0) − P(Ŷ=1|A=1)|`

In [ ]:
def selection_rate(yhat, A, a):
    return yhat[A == a].mean()

sr0 = selection_rate(yhat, A_te, 0)
sr1 = selection_rate(yhat, A_te, 1)
dp_gap = abs(sr0 - sr1)
ratio = min(sr0, sr1) / max(sr0, sr1)

print(f'selection_rate(A=0) = {sr0:.4f}')
print(f'selection_rate(A=1) = {sr1:.4f}')
print(f'DP_gap              = {dp_gap:.4f}')
print(f'80% rule ratio      = {ratio:.4f}  (regla EEOC: >= 0.80)')

## 3. Equal opportunity y equalized odds (Hardt-Price-Srebro 2016)

TPR = P(Ŷ=1 | Y=1, A=a) — `equal opportunity` exige TPR igual.  
FPR = P(Ŷ=1 | Y=0, A=a) — `equalized odds` exige TPR **y** FPR iguales.

In [ ]:
def tpr_fpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return tpr, fpr

tpr0, fpr0 = tpr_fpr(y_te[A_te == 0], yhat[A_te == 0])
tpr1, fpr1 = tpr_fpr(y_te[A_te == 1], yhat[A_te == 1])

eo_opp_gap = abs(tpr0 - tpr1)
eo_odds_gap = max(abs(tpr0 - tpr1), abs(fpr0 - fpr1))

print(f'TPR A=0 = {tpr0:.4f} | TPR A=1 = {tpr1:.4f}  -> equal_opportunity_gap = {eo_opp_gap:.4f}')
print(f'FPR A=0 = {fpr0:.4f} | FPR A=1 = {fpr1:.4f}')
print(f'equalized_odds_gap = max(|TPR_diff|, |FPR_diff|) = {eo_odds_gap:.4f}')

## 4. Calibration por grupo (Chouldechova 2017)

Para cada bin de score, P(Y=1 | Ŝ in bin) debe coincidir entre grupos.

In [ ]:
def calibration_curve_grouped(y_true, scores, A, a, n_bins=10):
    mask = A == a
    s, y = scores[mask], y_true[mask]
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(s, bins[1:-1])
    mean_score, mean_y = [], []
    for b in range(n_bins):
        m = idx == b
        if m.sum() > 0:
            mean_score.append(s[m].mean())
            mean_y.append(y[m].mean())
    return np.array(mean_score), np.array(mean_y)

ms0, my0 = calibration_curve_grouped(y_te, s, A_te, 0)
ms1, my1 = calibration_curve_grouped(y_te, s, A_te, 1)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect calibration')
ax.plot(ms0, my0, 'o-', label='A=0', color='steelblue')
ax.plot(ms1, my1, 's-', label='A=1', color='darkorange')
ax.set_xlabel('mean predicted score'); ax.set_ylabel('mean y_true')
ax.set_title('Reliability curves por grupo (baseline)')
ax.legend(); ax.grid(alpha=0.3); plt.show()

k = min(len(my0), len(my1))
cal_gap_baseline = np.abs(my0[:k] - my1[:k]).max()
print(f'calibration_gap baseline = {cal_gap_baseline:.4f}  (más bajo = más calibrado entre grupos)')

## 5. Demostración numérica del teorema de imposibilidad

Buscamos `t_0` y `t_1` tales que el modelo cumpla **DP exacta**. Recalculamos predictive parity (PPV) — debe **empeorar**.

In [ ]:
def apply_thresholds(s, A, t0, t1):
    return np.where(A == 0, (s >= t0).astype(int), (s >= t1).astype(int))

best = None
for t0 in np.linspace(0.1, 0.9, 81):
    target_sr = (s[A_te == 0] >= t0).mean()
    t1 = np.quantile(s[A_te == 1], 1 - target_sr)
    yhat_dp = apply_thresholds(s, A_te, t0, t1)
    acc = accuracy_score(y_te, yhat_dp)
    if best is None or acc > best['acc']:
        best = {'t0': t0, 't1': t1, 'acc': acc, 'yhat': yhat_dp}

print(f"DP-mitigated: t0={best['t0']:.3f}, t1={best['t1']:.3f}, acc={best['acc']:.4f}")
sr0_dp = selection_rate(best['yhat'], A_te, 0)
sr1_dp = selection_rate(best['yhat'], A_te, 1)
print(f'DP_gap post-fix = {abs(sr0_dp - sr1_dp):.4f}  (≈0 por construcción)')

def ppv(y_true, y_pred, A, a):
    m = (A == a) & (y_pred == 1)
    return y_true[m].mean() if m.sum() else float('nan')

ppv0_base = ppv(y_te, yhat, A_te, 0); ppv1_base = ppv(y_te, yhat, A_te, 1)
ppv0_dp = ppv(y_te, best['yhat'], A_te, 0); ppv1_dp = ppv(y_te, best['yhat'], A_te, 1)

print('\nPPV (predictive parity proxy) — debe ser igual entre grupos si hay calibración:')
print(f'  baseline:     PPV A=0={ppv0_base:.4f}, PPV A=1={ppv1_base:.4f}, gap={abs(ppv0_base-ppv1_base):.4f}')
print(f'  DP-mitigated: PPV A=0={ppv0_dp:.4f}, PPV A=1={ppv1_dp:.4f}, gap={abs(ppv0_dp-ppv1_dp):.4f}')
print('\n-> Forzar DP rompe predictive parity. Teorema KMR/Chouldechova 2017 en acción.')

## 6. Post-processing Hardt 2016: thresholds que minimizan equalized odds gap

In [ ]:
best_eo = None
for t0 in np.linspace(0.1, 0.9, 41):
    for t1 in np.linspace(0.1, 0.9, 41):
        yhat_eo = apply_thresholds(s, A_te, t0, t1)
        tpr0_, fpr0_ = tpr_fpr(y_te[A_te == 0], yhat_eo[A_te == 0])
        tpr1_, fpr1_ = tpr_fpr(y_te[A_te == 1], yhat_eo[A_te == 1])
        gap = max(abs(tpr0_ - tpr1_), abs(fpr0_ - fpr1_))
        acc = accuracy_score(y_te, yhat_eo)
        if best_eo is None or (gap < best_eo['gap'] - 1e-4) or (abs(gap - best_eo['gap']) < 1e-4 and acc > best_eo['acc']):
            best_eo = {'t0': t0, 't1': t1, 'gap': gap, 'acc': acc, 'yhat': yhat_eo}

print(f"EO-mitigated: t0={best_eo['t0']:.3f}, t1={best_eo['t1']:.3f}, EO_gap={best_eo['gap']:.4f}, acc={best_eo['acc']:.4f}")

## 7. Tabla comparativa

In [ ]:
def evaluate(yhat_, s_, y_te, A_te, name):
    sr0 = selection_rate(yhat_, A_te, 0); sr1 = selection_rate(yhat_, A_te, 1)
    tpr0, fpr0 = tpr_fpr(y_te[A_te == 0], yhat_[A_te == 0])
    tpr1, fpr1 = tpr_fpr(y_te[A_te == 1], yhat_[A_te == 1])
    return {
        'model': name,
        'accuracy': accuracy_score(y_te, yhat_),
        'AUC': roc_auc_score(y_te, s_),
        'DP_gap': abs(sr0 - sr1),
        'EO_opp_gap': abs(tpr0 - tpr1),
        'EO_odds_gap': max(abs(tpr0 - tpr1), abs(fpr0 - fpr1)),
        'PPV_gap': abs(ppv(y_te, yhat_, A_te, 0) - ppv(y_te, yhat_, A_te, 1)),
    }

tabla = pd.DataFrame([
    evaluate(yhat, s, y_te, A_te, 'baseline (t=0.5)'),
    evaluate(best['yhat'], s, y_te, A_te, 'DP-mitigated'),
    evaluate(best_eo['yhat'], s, y_te, A_te, 'EO-mitigated'),
])
print(tabla.round(4).to_string(index=False))

## Ejercicio guiado

1. Reemplazá el dataset sintético por **Adult Census** (UCI). Atributo protegido: `sex`. Target: `income > 50K`.
2. Implementá las 4 métricas a mano sobre el dataset real. ¿Cumple regla del 80%?
3. Repetí la mitigación EO con grid search. ¿Cuánto cae la accuracy?
4. Comparar tu implementación contra `fairlearn.metrics.MetricFrame` y `fairlearn.postprocessing.ThresholdOptimizer`.
5. Bonus: probá COMPAS (ProPublica) con `race`. Reproducí el debate ProPublica vs Northpointe — ¿calibración o equalized odds?

## Conclusiones

- **Demographic parity** ignora el ground truth; útil cuando las base rates son comparables.
- **Equal opportunity / equalized odds** (Hardt 2016) condicionan en Y — más defendibles en crédito, contratación, salud.
- **Calibration / predictive parity** (Chouldechova 2017) asegura que el score signifique lo mismo en cada grupo.
- **Teorema de imposibilidad** (KMR / Chouldechova 2017): si las base rates difieren, no se pueden tener las tres a la vez — hay que **elegir y documentar**.
- Post-processing con thresholds por grupo es la mitigación más simple y la que mejor revela el trade-off accuracy/fairness.

## ✅ Soluciones de los ejercicios

Un solo dataset sintético (atributo sensible `A`, target `y`, `score` de un modelo casi
calibrado) sirve para las 5 métricas. Todo con `numpy`/`sklearn`, sin internet. El hilo
conductor es el **teorema de imposibilidad**: forzar una noción de fairness rompe otra.

### Ejercicio 1 — Selection rate por grupo y regla del 80%

Entrenamos (simulamos) un score y umbralizamos en 0.5. Calculamos `P(Ŷ=1|A=0)`,
`P(Ŷ=1|A=1)`, el `DP_gap` y el ratio de la **regla del 80%** (EEOC): si la tasa del grupo
minoritario es <80% de la del mayoritario, hay *adverse impact*.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score

rng = np.random.default_rng(7)
N = 8000
A = rng.integers(0, 2, N)                              # atributo sensible (0/1)
q = rng.normal(np.where(A == 1, 0.4, -0.3), 1.0)       # aptitud latente: base rate distinto
p_true = 1 / (1 + np.exp(-q))
y = (rng.random(N) < p_true).astype(int)
# Score de un modelo razonable: sigmoide de q + ruido -> aprox calibrado en ambos grupos
score = 1 / (1 + np.exp(-(q + rng.normal(0, 0.5, N))))
yhat = (score > 0.5).astype(int)

def selection_rate(yh, A, a):
    return yh[A == a].mean()

sr0, sr1 = selection_rate(yhat, A, 0), selection_rate(yhat, A, 1)
dp_gap = abs(sr0 - sr1)
ratio80 = min(sr0, sr1) / max(sr0, sr1)
print("base rate real  y|A=0=%.2f  y|A=1=%.2f" % (y[A == 0].mean(), y[A == 1].mean()))
print("selection rate  A=0 %.3f  A=1 %.3f  DP_gap=%.3f  ratio80=%.2f" %
      (sr0, sr1, dp_gap, ratio80))
print("regla del 80%:", "FALLA" if ratio80 < 0.8 else "cumple")
assert dp_gap > 0.05, "hay disparidad de demographic parity por base rates distintas"
print("OK ejercicio 1 - DP gap y regla del 80% calculados")

### Ejercicio 2 — TPR y FPR por grupo (equalized odds)

Con la matriz de confusión **separada por grupo** calculamos
`equal_opportunity_gap = |TPR₀ − TPR₁|` y
`equalized_odds_gap = max(|ΔTPR|, |ΔFPR|)`.

In [ ]:
def tpr_fpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return tpr, fpr

tpr0, fpr0 = tpr_fpr(y[A == 0], yhat[A == 0])
tpr1, fpr1 = tpr_fpr(y[A == 1], yhat[A == 1])
equal_opportunity_gap = abs(tpr0 - tpr1)
equalized_odds_gap = max(abs(tpr0 - tpr1), abs(fpr0 - fpr1))
print("A=0  TPR=%.3f  FPR=%.3f" % (tpr0, fpr0))
print("A=1  TPR=%.3f  FPR=%.3f" % (tpr1, fpr1))
print("equal_opportunity_gap=%.3f  equalized_odds_gap=%.3f" %
      (equal_opportunity_gap, equalized_odds_gap))
assert equalized_odds_gap >= equal_opportunity_gap
print("OK ejercicio 2 - gaps de equal opportunity y equalized odds")

### Ejercicio 3 — Calibration curves por grupo

Binning de los scores en 10 bins. Para cada bin graficamos (numéricamente) `mean(y_true)`
vs `mean(score)`. Como el score es una sigmoide ruidosa de la aptitud, queda **calibrado en
ambos grupos**: las curvas caen sobre la diagonal aunque las tasas base difieran.

In [ ]:
def calibration_curve_grouped(y_true, s, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(s, edges) - 1, 0, n_bins - 1)
    xs, ys = [], []
    for b in range(n_bins):
        m = idx == b
        if m.sum() > 20:
            xs.append(s[m].mean()); ys.append(y_true[m].mean())
    return np.array(xs), np.array(ys)

x0, o0 = calibration_curve_grouped(y[A == 0], score[A == 0])
x1, o1 = calibration_curve_grouped(y[A == 1], score[A == 1])
ece0 = np.mean(np.abs(x0 - o0))   # desvio medio pred vs observado
ece1 = np.mean(np.abs(x1 - o1))
print("calibration gap medio (|pred-obs|)  A=0 %.3f  A=1 %.3f" % (ece0, ece1))
assert ece0 < 0.12 and ece1 < 0.12, "el score esta razonablemente calibrado en AMBOS grupos"
print("OK ejercicio 3 - calibracion ~diagonal en los dos grupos")

### Ejercicio 4 — Romper calibración forzando demographic parity

Elegimos umbrales por grupo `(t₀, t₁)` que igualan la selection rate (**DP exacta**). Con
las tasas base distintas, la **precisión (PPV)** entre predichos positivos deja de coincidir:
un "positivo" del modelo significa cosas distintas según el grupo. Es la demostración
numérica de la incompatibilidad DP ↔ predictive parity/calibration (Chouldechova 2017).

In [ ]:
target = 0.5 * (sr0 + sr1)                      # tasa comun objetivo

def thr_for_rate(s, rate):
    return np.quantile(s, 1 - rate)             # umbral que produce esa selection rate

t0 = thr_for_rate(score[A == 0], target)
t1 = thr_for_rate(score[A == 1], target)
pred_dp = np.where(A == 0, score > t0, score > t1).astype(int)
sr0d, sr1d = pred_dp[A == 0].mean(), pred_dp[A == 1].mean()

def ppv(y_true, yh):
    pos = yh == 1
    return y_true[pos].mean() if pos.sum() else float('nan')

ppv0, ppv1 = ppv(y[A == 0], pred_dp[A == 0]), ppv(y[A == 1], pred_dp[A == 1])
print("DP exacta:  sr A=0 %.3f  A=1 %.3f  (gap=%.3f)" % (sr0d, sr1d, abs(sr0d - sr1d)))
print("precision:  PPV A=0 %.3f  A=1 %.3f  (gap=%.3f)" % (ppv0, ppv1, abs(ppv0 - ppv1)))
assert abs(sr0d - sr1d) < 0.02, "forzamos demographic parity (misma selection rate)"
assert abs(ppv0 - ppv1) > 0.05, "...pero la precision por grupo diverge: DP rompe predictive parity"
print("OK ejercicio 4 - imposibilidad: DP exacta => precision desigual")

### Ejercicio 5 — Post-processing Hardt: minimizar equalized odds gap

Buscamos en grilla los umbrales `(t₀, t₁)` que **minimizan** el `equalized_odds_gap` y
reportamos el costo en accuracy global. Tabla comparativa: **baseline** (umbral único 0.5)
vs **DP-fixed** (ejercicio 4) vs **EO-fixed** (esta búsqueda).

In [ ]:
def eval_thresholds(a, b):
    pr = np.where(A == 0, score > a, score > b).astype(int)
    tpr0_, fpr0_ = tpr_fpr(y[A == 0], pr[A == 0])
    tpr1_, fpr1_ = tpr_fpr(y[A == 1], pr[A == 1])
    eo = max(abs(tpr0_ - tpr1_), abs(fpr0_ - fpr1_))
    dp = abs(pr[A == 0].mean() - pr[A == 1].mean())
    return dp, eo, accuracy_score(y, pr)

grid = np.linspace(0.2, 0.8, 25)
best = None
for a in grid:
    for b in grid:
        dp, eo, acc = eval_thresholds(a, b)
        if best is None or eo < best[1]:
            best = (dp, eo, acc, a, b)
t0_eo, t1_eo = best[3], best[4]

rows = [("baseline", 0.5, 0.5), ("DP-fixed", t0, t1), ("EO-fixed", t0_eo, t1_eo)]
print("%-10s %8s %8s %8s" % ("metodo", "DP_gap", "EO_gap", "acc"))
res = {}
for nm, a, b in rows:
    dp, eo, acc = eval_thresholds(a, b)
    res[nm] = (dp, eo, acc)
    print("%-10s %8.3f %8.3f %8.3f" % (nm, dp, eo, acc))
assert res["EO-fixed"][1] < res["baseline"][1], "EO-fixed reduce el equalized odds gap vs baseline"
print("OK ejercicio 5 - Hardt: minimiza EO gap; la tabla muestra el trade-off con accuracy")